# 06. Temporal Fusion Transformer (TFT)

## Background

The Temporal Fusion Transformer (Lim et al., 2019) is the most cited deep learning forecasting architecture, combining attention mechanisms with gating for multi-horizon forecasting with rich covariate information. TFT was designed specifically for real-world forecasting challenges: multiple prediction horizons, mixed input types, and interpretable attention weights.

TFT's key innovations:
- **Gated Residual Networks (GRN)**: conditional information suppression
- **Variable Selection Networks (VSN)**: learn which inputs matter per time step
- **Static Covariate Encoders**: propagate context from static features throughout
- **Multi-head attention**: captures long-range dependencies
- **Quantile outputs**: native probabilistic forecasting

## What You'll Learn

- TFT architecture decomposed component by component
- Gated Residual Network (GRN) as the fundamental building block
- Variable Selection Network: soft feature selection
- Quantile loss for probabilistic forecasting outputs

## Why This Matters

TFT is the production standard for multi-horizon business forecasting with covariates. It handles the realistic scenario where you have: known future inputs (planned promotions, holidays), static attributes (store size, product category), and historical observations — and need calibrated prediction intervals, not just point estimates.


## Gated Residual Network (GRN)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

class GatedLinearUnit(nn.Module):
    '''GLU: gate applied to linear transformation.
    GLU(x) = sigmoid(W2 x + b2) * (W1 x + b1)
    '''
    def __init__(self, d_model: int):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_model)
        self.fc2 = nn.Linear(d_model, d_model)

    def forward(self, x):
        return torch.sigmoid(self.fc2(x)) * self.fc1(x)

class GatedResidualNetwork(nn.Module):
    '''GRN: the core building block of TFT.
    GRN(a, c) = LayerNorm(a + GLU(eta1))
    where eta1 = W1 * eta2 + b1 + W3 * c
    and eta2 = ELU(W2 * a + b2)
    c is optional static context vector.
    '''

    def __init__(self, d_model: int, d_hidden: int, dropout: float = 0.1,
                 d_context: int = None):
        super().__init__()
        self.d_model = d_model

        self.fc1 = nn.Linear(d_model, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_model)

        if d_context is not None:
            self.context_proj = nn.Linear(d_context, d_hidden, bias=False)
        else:
            self.context_proj = None

        self.glu = GatedLinearUnit(d_model)
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, a: torch.Tensor, c: torch.Tensor = None) -> torch.Tensor:
        eta2 = F.elu(self.fc1(a))
        if c is not None and self.context_proj is not None:
            eta2 = eta2 + self.context_proj(c)
        eta1 = self.fc2(eta2)
        return self.layer_norm(a + self.glu(self.dropout(eta1)))

# Test GRN
D = 64
grn = GatedResidualNetwork(d_model=D, d_hidden=D*2, d_context=D)
x = torch.randn(8, 52, D)   # (batch, time, d_model)
c = torch.randn(8, D)        # (batch, d_model) static context
out = grn(x, c.unsqueeze(1).expand_as(x))
print(f'GRN: input {tuple(x.shape)} -> output {tuple(out.shape)}')
print(f'Parameters: {sum(p.numel() for p in grn.parameters()):,}')


## Variable Selection Network

In [ ]:
class VariableSelectionNetwork(nn.Module):
    '''VSN: learns soft weights over input variables.
    Each variable gets its own GRN for transformation,
    plus a shared GRN computes a softmax weight vector.
    '''

    def __init__(self, n_vars: int, d_model: int, d_hidden: int,
                 dropout: float = 0.1, d_context: int = None):
        super().__init__()
        self.n_vars = n_vars
        self.d_model = d_model

        # One GRN per variable
        self.var_grns = nn.ModuleList([
            GatedResidualNetwork(d_model, d_hidden, dropout)
            for _ in range(n_vars)
        ])

        # Softmax weight GRN
        self.weight_grn = GatedResidualNetwork(
            n_vars * d_model, d_hidden, dropout, d_context=d_context
        )
        self.weight_head = nn.Linear(n_vars * d_model, n_vars)

    def forward(self, x: torch.Tensor,
                context: torch.Tensor = None) -> tuple:
        # x: (B, T, n_vars, d_model) or (B, n_vars, d_model) for static
        B = x.shape[0]

        # Per-variable transformation
        var_outputs = []
        for i in range(self.n_vars):
            v = x[..., i, :]  # (B, T, d) or (B, d)
            var_outputs.append(self.var_grns[i](v))
        var_stack = torch.stack(var_outputs, dim=-2)  # (..., n_vars, d)

        # Softmax weights
        flat = x.reshape(*x.shape[:-2], -1)  # (..., n_vars*d)
        grn_out = self.weight_grn(flat, context)
        weights = torch.softmax(self.weight_head(grn_out), dim=-1)  # (..., n_vars)

        # Weighted combination
        combined = (var_stack * weights.unsqueeze(-1)).sum(dim=-2)  # (..., d)
        return combined, weights

vsn = VariableSelectionNetwork(n_vars=5, d_model=32, d_hidden=64)
x = torch.randn(8, 52, 5, 32)  # (batch, time, n_vars, d_model)
combined, weights = vsn(x)
print(f'VSN: input {tuple(x.shape)}')
print(f'  Combined: {tuple(combined.shape)}')
print(f'  Weights:  {tuple(weights.shape)} (softmax over {vsn.n_vars} variables)')
print(f'\nMean variable weights (should sum to 1):')
mean_weights = weights.mean(dim=(0, 1)).detach()
for i, w in enumerate(mean_weights):
    print(f'  Variable {i}: {w:.4f}')
print(f'  Sum: {mean_weights.sum():.4f}')


## Quantile Loss for Probabilistic Forecasting

In [ ]:
def quantile_loss(predictions: torch.Tensor, targets: torch.Tensor,
                   quantiles: list) -> torch.Tensor:
    '''Pinball loss for multiple quantiles.
    predictions: (B, H, n_quantiles)
    targets: (B, H)
    '''
    losses = []
    targets_expanded = targets.unsqueeze(-1).expand_as(predictions)

    for i, q in enumerate(quantiles):
        errors = targets_expanded[..., i] - predictions[..., i]
        # Asymmetric loss: penalizes under-prediction by q, over-prediction by (1-q)
        loss = torch.max((q - 1) * errors, q * errors)
        losses.append(loss.mean())

    return torch.stack(losses).mean()

class SimpleTFTHead(nn.Module):
    '''Multi-quantile output head — attaches to any encoder output.'''
    def __init__(self, d_model: int, horizon: int,
                 quantiles: list = [0.1, 0.5, 0.9]):
        super().__init__()
        self.quantiles = quantiles
        self.heads = nn.ModuleList([
            nn.Linear(d_model, horizon) for _ in quantiles
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, d_model) — last hidden state
        return torch.stack([head(x) for head in self.heads], dim=-1)
        # output: (B, horizon, n_quantiles)

QUANTILES = [0.1, 0.5, 0.9]
HORIZON = 12

head = SimpleTFTHead(d_model=64, horizon=HORIZON, quantiles=QUANTILES)
encoder_out = torch.randn(16, 64)
preds = head(encoder_out)  # (16, 12, 3)

targets = torch.randn(16, HORIZON) * 10 + 100
# Ensure monotonicity: q10 <= q50 <= q90
preds_sorted, _ = preds.sort(dim=-1)

loss = quantile_loss(preds_sorted, targets, QUANTILES)
print(f'Quantile head: predictions shape {tuple(preds.shape)}')
print(f'Quantile loss: {loss.item():.4f}')
print(f'\nCoverage check (in-sample):')
with torch.no_grad():
    q10 = preds_sorted[..., 0]
    q90 = preds_sorted[..., 2]
    coverage = ((targets >= q10) & (targets <= q90)).float().mean()
    print(f'  80% PI coverage: {coverage:.2%} (target: 80%)')
